### 53. Maximum Subarray
Given an integer array `nums`, find the subarray with the largest sum, and return its *sum*.
(A subarray is a contiguous non-empty sequence of elements within an array.)

**Example 1**:
```shell
Input: nums = [-2,1,-3,4,-1,2,1,-5,4]
Output: 6
```
Explanation: The subarray `[4,-1,2,1]` has the largest sum `6`.

**Example 2**:
```shell
Input: nums = [1]
Output: 1
```
Explanation: The subarray `[1]` has the largest sum `1`.

**Example 3**:
```shell
Input: nums = [5,4,-1,7,8]
Output: 23
```
Explanation: The subarray `[5,4,-1,7,8]` has the largest sum `23`.
 

**Constraints**:
```shell
1 <= nums.length <= 10^5
-10^4 <= nums[i] <= 10^4
```
**Follow up**: If you have figured out the $O(n)$ solution, try coding another solution using the **divide and conquer** approach, which is more subtle.

### Reasoning

#### Step 1 - Gather Requirements
In the integer array I need to find a subarray, a contiguous sequence of items, that add up to the largest sum within the input array.

**A subarray** is a sequence of adjacent input items.

**A subsequence** is a sequence of input items that **keeps the original relative order** but can skip elements.

**A subset** is a sequence of elements in **any order**.

```shell
nums = [3, -1, 4, 2]
  subarray:     [-1, 4]      ← adjacent, same order
  subsequence:  [3, 4]       ← same order, gaps allowed
  NOT either:   [4, 3]       ← order changed
```

> That difference is the reason this problem is hard. Because a subarray has to be **contiguous**, you can't pick out all the positive numbers and add them.

1. Is the input array sorted?
2. Does the order of the output matter? Is it the sum (integer), the subarray or its indices?
3. Can input be empty? Is an empty subarray (sum = 0) a valid answer?
4. Are there duplicates and negatives withing the input array? Can all of them be negative?

---

1. **Sorted?** No. Assume it is in arbitrary order.
2. **Output?** Return only the **sum** as an `int`. Your question about whether output order matters doesn't apply to a single number. Save it for problems that return collections.
3. **Empty?** `1 <= len(nums) <= 10^5`, so the input is never empty. The subarray must be **non-empty**, so a sum of 0 from an empty subarray is **not** a valid answer.
4. **Duplicates and negatives?** Both are possible. Values range over `-10^4 <= nums[i] <= 10^4`. **Yes, every value can be negative.**

One question you could add: _"Can the sum overflow?"_ In Python it can't, but saying it out loud shows you're aware of the issue in Java or C++.

Answers 3 and 4 together matter more than they look. Keep them in mind for Step 2.

---

#### Step 2 - Example + Edge Cases
1. `[-2, 1, -3, 4, -1, 2, 1, -5, 4]` returns `[4, -1, 2, 1]` (sum = 6). After `[-2, 1, -3]` with sum = `-4` subarray up to `-5` is what makes the biggest sum.
2. `[5]` returns `[5]` since it's the only item sum = `5`
3.  `[-3, -1, -2]` returns `[-1]` since it produces the biggest sum -1
4. `[5, 4, -1, 7, 8]` returns `[5, 4, -1, 7, 8]` since -1 doesn't drop too much the total sum of the whole array, which is 23.

- In example 1 you wrote that after `[-2, 1, -3]` (sum −4), the best subarray starts fresh. **Why** is starting fresh better than carrying that prefix forward? Answer in one sentence about what a prefix with a sum of −4 does to anything you attach it to.
- In example 4 you wrote that "−1 doesn't drop the sum too much." Compare it with example 1, where you _did_ drop elements. Ask about the running total **before** the −1, not about the −1 itself. What is the rule that decides "keep going" versus "cut here"?

```shell
negative prefix = -4,  next element = 100

carry it:    -4 + 100 = 96
start fresh:       100 = 100   ← always better
```
A big number coming next **doesn't change** the conclusion. Whatever comes next, adding a negative prefix to it **gives a smaller sum** than leaving the prefix off. So the rule, completed:
> **Rule 1**: If the **running sum** before the current element is **negative**, **drop it** and start over at the current element. Otherwise, keep extending.

![Rule](../../images/53max_subarray_rule.png)

This rule is the key idea behind the optimized solution. We'll use it in Step 4.

**What should `max_sum` start as, so that it can never beat any real subarray sum?** (There are two common choices. One of them uses the input itself.)
```shell
max_sum = nums[0]         ← uses the input
max_sum = float('-inf')   ← a value smaller than every possible number
```
`nums[0]` is the stronger choice because `[nums[0]]` is itself a real subarray. The starting value is therefore a **valid candidate answer**, not a placeholder. It's $O(1)$ and needs no proof of safety. `-inf` also works, but on an empty input it would return `-inf`, and our constraints rule out empty input anyway.

```shell
[-2, 1, -3, 4, -1, 2, 1, -5, 4]  → 6
[5]                              → 5
[-3, -1, -2]                     → -1   (catches the max_sum = 0 bug)
[5, 4, -1, 7, 8]                 → 23
```

---

#### Step 3 - Brute Force

```shell
[5, 4, -1, 7, 8]
start 0 -> stop 0 -> [5] -> sum 5
start 0 -> stop 1 -> [5, 4] -> sum 9
start 0 -> stop 2 -> [5, 4, -1] -> sum 8
start 0 -> stop 3 -> [5, 4, -1, 7] -> sum 15
```
![Visual](../../images/53max_subarray_brute1.png)
- Two boundaries, `start` and `stop`, define **every subarray**, and the outer two loops list all of them.
- The innermost loop computes the **running sum** of the **current subarray** - `start..stop` from scratch each time, up to `n` items, which makes it $O(n³)$.

In [ ]:
def max_subarr1(nums: list[int]) -> int:
    # brute force - check every possible subarrays - 2 nested loops - O(n^3)
    # setup
    # set max sum to the first element: max_sum = nums[0]
    # set start of subarray: for loop - full pass
    # set stop of subarray: nested for loop - [start, len(nums)]

    # 1. set current subbarray sum: current_sum = 0
    # 2. compute current subbarray sum: current_sum += nums[i]
    # 3. write max_sum: max(max_sum, current_sum)
    # 4. return max_sum

    max_sum = nums[0]

    for start in range(len(nums)):  # set start - O(n)
        for stop in range(start, len(nums)):  # set stop - O(n)
            current_sum = 0

            # makes time O(n^3)
            # redunant work - re-scanning subarray rather than RE-USING the prefix
            for i in range(start, stop + 1):  # compute surrent sum between start & stop
                current_sum += nums[i]

            max_sum = max(max_sum, current_sum)  # get bigger sum

    return max_sum

In [9]:
max_subarr1([-2, 1, -3, 4, -1, 2, 1, -5, 4])  # 6
# max_subarr1([5])  # 5
# max_subarr1([-3, -1, -2])  # -1   (catches the max_sum = 0 bug)
# max_subarr1([5, 4, -1, 7, 8])  # 23

6

- **Time**: $O(n^3)$ cubic - for `n` items setting the **start** of the subarray there is `n` total work to set the **stop** boundary for the subarray and `n` operations to **sum the subarray items**.
- **Space**: $O(1)$ constant, no data structures are used to store aside from the input.

The number of stops **shrinks** as `start` moves right. That shrinking count is the reason the arithmetic series formula applies:
```shell
stops per start: n + (n-1) + ... + 1 = n(n+1)/2  → O(n²) pairs
```
"At most `n` stops per start" is a valid **upper bound**, and it gives the same Big-O. But "`n` stops for every start" and "arithmetic series" **can't both be true**. Use the series, since it's exact.

1. The `+ 1` at the end of the series
It's the **last term**, the number of stops for the last start:
```shell
nums has n = 9 elements (indices 0..8)

start 0 → stops 0..8 → 9 = n
start 1 → stops 1..8 → 8 = n-1
...
start 7 → stops 7..8 → 2
start 8 → stops 8..8 → 1   ← the "+ 1": only [nums[8]] itself
```
The series stops at 1, not 0, because every start has at least one stop: the subarray of that single element.

2. The `(n + 1)` in the formula
It comes from pairing the terms, which is the Gauss trick. Write the sum forwards and backwards, then add the columns:
```shell
  9 + 8 + 7 + 6 + 5 + 4 + 3 + 2 + 1
+ 1 + 2 + 3 + 4 + 5 + 6 + 7 + 8 + 9
  ─────────────────────────────────
 10 +10 +10 +10 +10 +10 +10 +10 +10   = 9 columns × 10 = 90
```
- Every column sums to **n + 1**: the biggest term plus the smallest, `n + 1`, then `(n-1) + 2`, and so on.
- There are `n` columns.
- You wrote the sum **twice**, so divide by 2.
```shell
n(n + 1) / 2  →  9 × 10 / 2 = 45
```
The 45 matches the count from before.

**Summary**

`max_subarr1` — $O(n³)$ time, $O(1)$ space
- `start`/`stop` pairs: `n + (n-1) + ... + 1 = n(n+1)/2 →` $O(n²)$
- Each pair **re-sums `nums[start..stop]` from scratch** → up to `n` more work
- Total: $O(n²)$ pairs $×$ $O(n)$ per `sum =` $O(n³)$
- **Redundant work**: `sum(start..stop) = sum(start..stop-1) + nums[stop]` - **3rd nested loop** re-scans & **recomputes the whole prefix** instead of reusing it

> **Fix 1**: keep a **running sum PER START**, extend by one element.

![Visual](../../images/53max_subarray_brute2.png)

In [ ]:
def max_subarr2(nums: list[int]) -> int:
    # brute force 2 - O(n^2)
    # setup
    # set max sum: max_sum = nums[0]
    # set start: for loop - full scan
    # set current running sum PER START
    # set stop: for loop - [start, end]

    # 1. compute current running sum for start/stop subarray: current_sum += nums[stop]
    # 2. write max sum: max(max_sum, current_sum)
    # 3. return max_sum

    max_sum = nums[0]

    for start in range(len(nums)):  # O(n)
        current_sum = 0

        for stop in range(start, len(nums)):  # O(n)
            current_sum += nums[stop]

            max_sum = max(max_sum, current_sum)

    return max_sum

In [7]:
# max_subarr2([-2, 1, -3, 4, -1, 2, 1, -5, 4])  # 6
# max_subarr2([5])  # 5
# max_subarr2([-3, -1, -2])  # -1   (catches the max_sum = 0 bug)
max_subarr2([5, 4, -1, 7, 8])  # 23

23

- **Time**: $O(n^2)$ quadratic. The nested loop performs `n` ops as we stated before `n + (n - 1) + ... + 1 = n(n + 1)/2` - **`n` times `(n + 1)/2`**, which makes the time quadratic. So we got rid of the inner-most loop and directly count & compare the current running sum vs previously computed max sum.
- **Space**: $O(1)$ constant.
```shell
max_subarr2
Time:  O(n²) — n(n+1)/2 start/stop pairs, each handled in O(1)
               (one addition + one comparison, no rescanning)
Space: O(1)  — only two scalars (max_sum, current_sum),
               nothing that grows with n

vs max_subarr1: O(n³) — same O(n²) pairs, but O(n) to sum each one
```

---

**Fix Dropping the Prefix**

What it still wastes happens **between starts**: every time `start` moves, the running sum is **thrown away** and **rebuilt** from zero.

Combine that with the rule you found earlier: **drop the running sum only when it's negative**.

**Here's the one question**: do we ever need to restart from scratch at every index, or only at specific moments? If only at specific moments, which ones?
- Given the rule, we **only restart from scratch** when the current sum is **negative**. That is, when the current sum is negative - reset it to `0` and start computing from the **current** element.

> **The redundant work** is **re-scanning the prefix you already summed** one step earlier.

> **Fix 2**: keep one running sum for the whole pass and reset it only when it's negative.

![Visual](../../images/53max_subarray_kaden.png)

#### Step 4 - Select Algorithm
**Kadane's algorithm**: one pass, **carrying forward the previous current sum** adding it to the next item in the array.

> **Key insight**: A negative running sum **can only hurt** whatever follows it, so drop it and start over at the current element.

```shell
[-2, 1, -3, 4, -1, 2, 1, -5, 4]

idx  num  current_sum   max_sum
 0   -2      -2           -2    ← init from nums[0]
 1    1       1  restart   1    ← prev sum -2 < 0, drop it
 2   -3      -2            1    ← -2 < 1, no update
 3    4       4  restart   4    ← prev sum -2 < 0, drop it
 4   -1       3            4    ← 3 < 4, no update
 5    2       5            5
 6    1       6            6
 7   -5       1            6    ← 1 < 6, no update
 8    4       5            6    ← 5 < 6, no update
                          ───
                    answer: 6 ✓
```

#### Step 5, 6 - Signature + Skeleton
```py
def max_subarr3(nums: list[int]) -> int:
    """Kaden's algorithm - track the running sum, re-use it forward & discard it once it's negative"""
    # Kaden's - carry the running sum forward, discard it once it's negative
    # setup
    # set max sum to FIRST element: max_sum = nums[0]
    # set current running sum to 0: current_sum = 0
    # scan input - for loop - full pass

    # 1. test: if current sum is negative - drop it, reset to 0
    # 2. increment the current running sum
    # 3. write max sum: max(max_sum, current_sum)
    # 4. return max_sum
```

#### Step 7 - Write Code

**Is the Prefix Hurting Me?**

In [ ]:
def max_subarr3(nums: list[int]) -> int:
    # Kadane's algorithm - one pass - carry running sum forward
    # setup
    # set max sum to the first element: max_sum = nums[0]
    # set current sum to 0: current_sum = 0
    # scan input - for loop - full pass

    # 1. test: current sum < 0 (negative prefix):
    #   drop it - remove that portion from current sum: current_sum = 0
    # 2. compute current sum: current_sum += nums[i]
    # 3. write max sum: max_sum = max(max_sum, current_sum)
    # 4. reuturn max sum

    max_sum = nums[0]
    current_sum = 0  # why nums[0] don't work here?

    for num in nums:
        # drop prefix if it's hurmful
        if current_sum < 0:
            current_sum = 0

        current_sum += num
        max_sum = max(max_sum, current_sum)

    # max_sum = current_sum = nums[0]

    # for i in range(1, len(nums)):
    #     if current_sum < 0:
    #         current_sum = 0

    #     current_sum += nums[i]
    #     max_sum = max(current_sum, max_sum)

    return max_sum

#### Step 8 - Trace Examples + Edge Cases

In [5]:
# max_subarr3([-2, 1, -3, 4, -1, 2, 1, -5, 4])  # 6
# max_subarr3([5])  # 5
# max_subarr3([-3, -1, -2])  # -1   (catches the max_sum = 0 bug)
max_subarr3([5, 4, -1, 7, 8])  # 23

28

#### Step 9 - Complexity Analysis

`current_sum` changes on every step. `max_sum` only changes when `current_sum` beats it. At index 7, `current_sum` drops to 1 while `max_sum` stays at 6. The best answer seen so far is never lost, even when the current run gets worse.

**Kadane's - Is Prefix Hurting?**
- **Time**:  $O(n)$ — each element visited exactly once, $O(1)$ work per element
- **Space**: $O(1)$ — two scalars, independent of input size

**vs `max_subarr2`**: $O(n²)$ → $O(n)$, since we never reset at every start; we only reset when the running sum goes **negative**

#### Step 10 - Test + Dubug

In [ ]:
import pytest


@pytest.mark.parametrize("nums, expected", [
    pytest.param([-2, 1, -3, 4, -1, 2, 1, -5, 4], 6, id="general_case"),
    pytest.param([5], 5, id="single_element"),
    pytest.param([-3, -1, -2], -1, id="extreme_all_negative"),
    pytest.param([5, 4, -1, 7, 8], 23, id="extra_case"),
])
def test_max_subarr3_general_and_extreme(nums, expected):
    assert max_subarr3(nums) == expected

#### Add Indexes
![Visual](../../images/53max_subarray_idx_return.png)

In [ ]:
def max_subarr4(nums: list[int]) -> tuple[int, int, int]:
    # separating run_start from best_start is what made it work
    # Kadane's + indices for best sum subarray
    # setup
    # set max & current running sum to the first element
    # set 3 index trackers: 1) current running start; 2) best start; 3) best end to 0
    # scan input - for loop - [1, end]

    # 1. test: "Does the Prefix hurt me?" - yes -> drop it; current_sum < 0 -> current_sum = 0
    #    track subarray start index: update current running start to the present index: run_start = i (start over)
    # 2. increment current sum: current_sum += nums[i]
    # 3. test: "Is current sum bigger max sum?" - yes -> update max sum to current sum
    #    track best subarray start & end indices: update best start index to current run start index
    #    (it points to the current subarray start) & best end to the current index
    #    (it points to the current subarray end)
    # 4. return max_sum

    max_sum = current_sum = nums[0]

    run_start = best_start = best_end = 0

    for i in range(1, len(nums)):
        # drop negative prefix + reset subarray start
        if current_sum < 0:
            current_sum = 0
            run_start = i  # current run begins

        # compute current running subarray's sum
        current_sum += nums[i]

        if current_sum > max_sum:
            max_sum = current_sum

            # update best subarray's start & end
            best_start = run_start
            best_end = i


    return (max_sum, best_start, best_end)

In [3]:
# index tracking testing
max_subarr4([-1, 3, -1, 2])  #   → (4, 1, 3)  ✓
max_subarr4([-3, -1, -2])  #     → (-1, 1, 1) ✓  run_start resets every step; only idx 1 sets a record
max_subarr4([5, -10, 1])  #      → (5, 0, 0)  ✓  run_start moves to 2, but best_start stays 0

(5, 0, 0)

**That last case shows why two variables are needed. `run_start` moved and the saved answer stayed put**.

[-1, 3, -1, 2]

```shell
idx  num  current_sum  max_sum  run_start  best_start  end
 0   -1       -1          -1        0          0        0   ← init
 1    3        3           3        1          1        1
 2   -1        2           3        1          1        1
 3    2        4           4        1          1        3
```

In [ ]:
import pytest


@pytest.mark.parametrize("nums, expected",
    [
        pytest.param([-2, 1, -3, 4, -1, 2, 1, -5, 4], (6, 3, 6), id="general_case"),
        pytest.param([5], (5, 0, 0), id="single_element"),
        pytest.param([-3, -1, -2], (-1, 0, 0), id="extreme_all_negative"),
        pytest.param([5, 4, -1, 7, 8], (23, 0, 4), id="extra_case"),
        pytest.param([-1, 3, -1, 2], (4, 1, 3), id="extra_idx_case1"),
        pytest.param([-3, -1, -2], (-1, 1, 1), id="extra_idx_case2"),
        pytest.param([5, -10, 1], (5, 0, 0), id="extra_idx_case3"),
    ]
)
def test_max_subarr4_general_and_extreme(nums, expected):
    assert max_subarr4(nums) == expected

### Dynamic Programming Version

In [1]:
def max_subarr5(nums: list[int]) -> tuple[int, int, int]:
    # add indexes of the best subarray
    max_sum = current_sum = nums[0]

    start = end = 0
    temp_start = 0

    for i in range(1, len(nums)):

        if nums[i] > current_sum + nums[i]:
            current_sum = nums[i]
            temp_start = i
        else:
            current_sum += nums[i]

        if current_sum > max_sum:
            max_sum = current_sum
            start = temp_start
            end = i

    return max_sum, start, end

In [2]:
# index tracking testing
max_subarr5([-1, 3, -1, 2])  #   → (4, 1, 3)  ✓
max_subarr5([-3, -1, -2])  #     → (-1, 1, 1) ✓  run_start resets every step; only idx 1 sets a record
max_subarr5([5, -10, 1])  #      → (5, 0, 0)  ✓  run_start moves to 2, but best_start stays 0

(5, 0, 0)

Read it as: _"the best subarray **ending at `i`** either starts fresh at `i` or extends the best one ending at `i−1`."_

That's a **dynamic programming** recurrence: the answer at `i` is built from the answer at `i−1`. Kadane's is often taught as the simplest DP problem for this reason. When you reach DP in the progression map, this line will come back.

One small thing they share: both start the loop at `range(1, ...)` with `current_sum = nums[0]`. That's cleaner than your earlier `max_subarr3`, which started `current_sum` at 0 and looped from index 0. Both setups work, but in this one the initialization of `max_sum` and `current_sum` describes the same real subarray, `[nums[0]]`.

**Summary:** use `max_subarr4` when explaining, and recognize `max_subarr5` as the DP form of the same rule.